# Basic Prompt Structures

## Overview

How you *structure* a prompt matters as much as what you put in it. This notebook focuses on the two fundamental architectural patterns every prompt engineer needs to master: the **single-turn prompt** — a self-contained instruction that asks for everything in one shot — and the **multi-turn prompt** — a sequence of exchanges where each message builds on the last.

These are not just technical distinctions. They represent two fundamentally different models of how you communicate with an AI, and choosing the wrong one for a given task is one of the most common reasons prompts underperform.

**Tools used in this notebook:** `openai` · `langchain` · `python-dotenv`

---

## Motivation

Most people start with single-turn prompts because they mirror how we use search engines: one input, one output, done. That works for simple queries. It breaks down the moment your task requires context, nuance, or back-and-forth refinement.

Multi-turn prompts solve this — but they introduce their own complexity. How do you maintain context across exchanges? What happens when the conversation grows long? How do you structure a dialogue so the model stays on track rather than drifting?

This notebook builds the mental model and the practical toolkit for both approaches, so you can make a deliberate choice about which structure serves your task rather than defaulting to whichever one feels familiar.

---

## Key Components

**Single-turn prompts** are self-contained interactions — the prompt includes everything the model needs to respond, and the exchange ends there. They are fast, simple, and effective for well-defined tasks that don't require iteration.

**Multi-turn prompts** are structured conversations where meaning accumulates across messages. The model's previous responses become part of the context for everything that follows, enabling tasks that require clarification, refinement, or sequential reasoning.

**Prompt templates** are parameterized structures that separate the fixed logic of a prompt from its variable inputs. Rather than rewriting a prompt for every new use case, you define the pattern once and substitute values as needed — making your prompts reusable, testable, and scalable.

**Conversation chains** are LangChain's mechanism for managing multi-turn dialogue. They handle the bookkeeping of conversation history automatically, so you can focus on the content of each exchange rather than manually threading context from one message to the next.

---

## Method Details

The notebook moves from simple to complex: starting with bare single-turn API calls, introducing templates to make them reusable, then building out into multi-turn conversations and managed conversation chains. Each section pairs a conceptual explanation with a working code example, and key sections include direct comparisons — the same task handled with and without structure — so the practical difference is concrete and observable rather than theoretical.

## Setup

First, let's import the necessary libraries and set up our environment.

In [16]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_core.prompts import ChatPromptTemplate

In [11]:
# 1. Load the .env file
# This automatically puts variables into os.environ
load_dotenv() 

# 2. Check if the key exists (Fail Fast)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing! Check your .env file.")

# 3. Initialize (LangChain automatically looks for the env var)
llm = ChatOpenAI(model="gpt-4o-mini")

## 1. Single-Turn Prompts

A single-turn prompt is the most atomic unit of interaction with a language model — one input, one response, done. There is no back-and-forth, no memory of what came before, and no opportunity to clarify after the fact. The model reads your prompt, generates a completion, and stops.

This simplicity is both its strength and its constraint. Single-turn prompts are fast, lightweight, and easy to deploy at scale — a single well-crafted template can power thousands of independent requests without any conversational overhead. But because the model has only one shot at understanding your intent, everything depends on how clearly and completely that intent is expressed upfront.

The practical implication: **what you don't say, the model will assume.** Every gap in your prompt is filled by the model's best statistical guess based on its training data. Sometimes that guess aligns with what you wanted. Often it doesn't — and the only way to close the gap is to make your instructions more explicit.

Single-turn prompts are the foundation of everything else in this curriculum. Before you can chain prompts together, assign roles, or build reasoning pipelines, you need to be able to write a single prompt that reliably produces what you want. That skill — writing complete, unambiguous, well-scoped instructions — is what this section builds.

In [12]:
single_turn_prompt = "What are the three primary colors?"
print(llm.invoke(single_turn_prompt).content)

The three primary colors are red, blue, and yellow. These colors cannot be created by mixing other colors together and serve as the foundation for creating a wide range of other colors through mixing. In the RGB color model used in digital displays, the primary colors are red, green, and blue.


Now, let's use a PromptTemplate to create a more structured single-turn prompt:

In [13]:
structured_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide a brief explanation of {topic} and list its three main components."
)

chain = structured_prompt | llm
print(chain.invoke({"topic": "color theory"}).content)

Color theory is a framework that explains how colors interact, how they can be combined, and how they can be perceived. It is used in various fields like art, design, and photography to create aesthetically pleasing compositions and to convey emotions or messages through color.

The three main components of color theory are:

1. **Hue**: This refers to the name of a color (such as red, blue, green, etc.) and is the characteristic that distinguishes one color from another.

2. **Saturation**: Also known as chroma, saturation describes the intensity or purity of a color. A highly saturated color appears vibrant and vivid, while a less saturated color looks more muted or grayish.

3. **Value**: This component refers to the lightness or darkness of a color. It is determined by the amount of light that a color reflects, and it plays a crucial role in creating contrast and depth in visual compositions.

These components work together to create a vast spectrum of colors and help artists and d

## 2. Multi-Turn Prompts (Conversations)

Where single-turn prompts are self-contained, multi-turn prompts are cumulative. Each exchange builds on the last — the model carries forward the context of everything said so far and uses it to shape every subsequent response. This is what makes multi-turn interactions feel less like querying a database and more like collaborating with a thinking partner.

But there's a critical detail that separates how this *feels* from how it actually *works*: the model has no persistent memory. What appears to be an ongoing conversation is, under the hood, the entire conversation history being re-sent to the model with every new message. The model isn't remembering — it's re-reading. This distinction matters enormously in practice, because it means context can be managed, manipulated, and engineered just like any other part of the prompt.

This opens up capabilities that single-turn prompts simply can't achieve. You can progressively refine an output across multiple exchanges — asking the model to draft, then revise, then reformat, then adapt for a different audience — without ever restarting from scratch. You can steer a reasoning process mid-flight, pushing back on a conclusion or asking the model to reconsider a specific assumption. You can build up complex context incrementally, introducing constraints and requirements in layers rather than all at once.

The tradeoff is complexity. Multi-turn interactions require you to think not just about what you're asking *now*, but about how earlier exchanges are shaping the model's interpretation of your current message. A careless early prompt can anchor the model to a frame that's difficult to escape later. A well-engineered conversation, by contrast, uses each turn intentionally — guiding the model step by step toward an output that a single prompt could never have produced alone.

In [19]:
# 2. Define the Prompt with a History placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# 3. Create the Chain
chain = prompt | llm

# 4. Manage History manually (The Modern standard)
chat_history = []

def ask_question(user_input):
    # Invoke the chain passing the current history list
    response = chain.invoke({
        "input": user_input,
        "chat_history": chat_history
    })
    
    # Update history with the new exchange
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response.content))
    
    return response.content

# Your specific workflow:
print(ask_question("Hi, I'm learning about space. Can you tell me about planets?"))
print(ask_question("What's the largest planet in our solar system?"))
print(ask_question("How does its size compare to Earth?"))

Of course! Planets are celestial bodies that orbit stars, including our Sun. They come in various sizes, compositions, and characteristics, and they can be categorized into different types. Here's an overview of the main categories of planets in our solar system:

### 1. **Terrestrial Planets**
These are rocky planets that are closer to the Sun. They have solid surfaces and are composed mainly of rock and metal. The terrestrial planets in our solar system are:

- **Mercury:** The closest planet to the Sun, it has extreme temperature variations and a very thin atmosphere.
- **Venus:** Similar in size and composition to Earth, it has a thick atmosphere primarily composed of carbon dioxide, leading to a strong greenhouse effect and very high temperatures.
- **Earth:** The only known planet to support life, it has liquid water, a diverse atmosphere, and a stable climate.
- **Mars:** Known as the Red Planet, it has a thin atmosphere and evidence of past water. It has seasons and polar ice c

Let's compare how single-turn and multi-turn prompts handle a series of related questions:

In [21]:
# Single-turn prompts (No change needed, but invoke is standard)
prompts = [
    "What is the capital of France?",
    "What is its population?",
    "What is the city's most famous landmark?"
]

print("--- Single-turn responses ---")
for prompt in prompts:
    # Use .invoke() instead of calling the object directly
    response = llm.invoke(prompt)
    print(f"Q: {prompt}\nA: {response.content}\n")

--- Single-turn responses ---
Q: What is the capital of France?
A: The capital of France is Paris.

Q: What is its population?
A: Could you please specify which location or entity you are referring to for the population estimate?

Q: What is the city's most famous landmark?
A: To provide you with the best answer, could you please specify which city you are referring to?



In [22]:
# Multi-turn prompts (The Modern LCEL Way)
print("--- Multi-turn responses ---")

# 1. Define the template with a placeholder for history
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# 2. Create the chain using the pipe operator
chain = prompt_template | llm

# 3. Manage the state (history) manually or with a simple list
history = []

for p in prompts:
    # Invoke the chain with the current history
    response = chain.invoke({"input": p, "chat_history": history})
    
    # Update the history with the exchange
    history.append(HumanMessage(content=p))
    history.append(AIMessage(content=response.content))
    
    print(f"Q: {p}\nA: {response.content}\n")

--- Multi-turn responses ---
Q: What is the capital of France?
A: The capital of France is Paris.

Q: What is its population?
A: As of my last knowledge update in October 2021, the population of Paris was approximately 2.1 million people within the city limits. However, if you include the metropolitan area, the population is about 11 million. For the most current population figures, I recommend checking the latest statistics from a reliable source.

Q: What is the city's most famous landmark?
A: The most famous landmark in Paris is the Eiffel Tower. It is an iconic symbol of the city and is visited by millions of tourists every year. The tower was completed in 1889 and stands at 1,083 feet (330 meters) tall.



## Conclusion

Prompt structure is not a stylistic choice — it is an architectural one. The decision between a single-turn and multi-turn approach, between a hardcoded string and a reusable template, between an unmanaged conversation and a deliberately engineered one, shapes what the model can and cannot do for you before you've written a single word of actual content.

What this notebook has established is the foundation: single-turn prompts for discrete, self-contained tasks where clarity and completeness must live entirely within one instruction; multi-turn conversations for complex, iterative work where context accumulates and each exchange builds toward something a single prompt couldn't reach alone; PromptTemplates for transforming one-off instructions into scalable, testable, reusable systems; and conversation chains for managing the context window deliberately rather than letting it grow unchecked.

These are not four separate techniques — they are four expressions of the same underlying principle: **structure produces reliability**. The more intentionally you design the shape of your interaction with a model, the more predictable and useful its outputs become.

This is the mindset every subsequent notebook in this curriculum assumes. From here, the techniques grow more sophisticated — chains, reasoning pipelines, optimization loops, safety layers — but they all operate on the same raw material you've been working with here: a prompt, a model, and the gap between what you intend and what you communicate. Closing that gap, systematically and at scale, is the entire discipline.